In [4]:
import pandas as pd
import seaborn as sns
import math
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import numpy as np
import os
import sys
import glob
from imageio import volread as imread
from skimage.filters import threshold_otsu
from skimage import measure
from scipy import stats
import umap
from sklearn.decomposition import PCA
import math
from scipy import stats
from scipy.stats import pearsonr
import pickle
from scipy.spatial.distance import euclidean

from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, classification_report


In [5]:
import pickle
import os

embedding_path = "/scr/vidit/neural_features/neuron_feature_extraction/DINO"

with open(os.path.join(embedding_path, "train_embeddings.pkl"), 'rb') as f:
    train_real_embeddings = pickle.load(f)

with open(os.path.join(embedding_path, "test_embeddings.pkl"), 'rb') as f:
    test_real_embeddings = pickle.load(f)

In [6]:
# Get all the embeddings in a dimension of (N, 384*num_channels)

train_embeddings = [] 
test_embeddings = []

train_genes = []
test_genes = []
for emb in train_real_embeddings:
    if emb['metadata']['Gene'][0] == 'DGKE' or emb['metadata']['Gene'][0] == 'GAS7':
        continue
    if emb['metadata']['Time'][0] != 'D28':
        continue
    #print(emb['embedding'].shape)  # should be (384*num_channels,)
    train_embeddings.append(emb['embedding'].reshape(1, -1))
    train_genes.append(emb['metadata']['Gene'])
for emb in test_real_embeddings:
    if emb['metadata']['Gene'][0] == 'DGKE' or emb['metadata']['Gene'][0] == 'GAS7':
        continue
    if emb['metadata']['Time'][0] != 'D28':
        continue
    test_embeddings.append(emb['embedding'].reshape(1, -1))
    test_genes.append(emb['metadata']['Gene'])

train_embeddings = np.array(train_embeddings)  # shape (N, 384*num_channels)
train_embeddings = train_embeddings.squeeze()  # shape (N, 384*num_channels)

test_embeddings = np.array(test_embeddings)  # shape (N, 384*num_channels)
test_embeddings = test_embeddings.squeeze()  # shape (N, 384*num_channels)
print(train_embeddings.shape, test_embeddings.shape)
print(len(train_genes), len(test_genes))

(1430, 172032) (352, 172032)
1430 352


In [7]:
# Get all the embeddings per image, and then get the first 20 PCA columns per channel

from tqdm import tqdm

per_channel_pca = np.array([14, 20]) # Size of the following that 14 is the number of channels and 25 PCA columns
pca_embeddings_train = None # 280 is num_chans * PCA columns
pca_embeddings_test = None

for chan_num in range(14):
    channel_embeddings = train_embeddings[:, chan_num*384:(chan_num+1)*384]  # shape (N, 384)
    pca = PCA(n_components=20)
    pca_model = pca.fit(channel_embeddings)
    train_channel_pca = pca_model.transform(channel_embeddings)  # shape (N, 20)
    test_channel_pca = pca_model.transform(test_embeddings[:, chan_num*384:(chan_num+1)*384])  # shape (N, 20)
    
    if pca_embeddings_train is None:
        pca_embeddings_train = train_channel_pca
        pca_embeddings_test = test_channel_pca
    else:
        pca_embeddings_train = np.concatenate((pca_embeddings_train, train_channel_pca), axis=1)
        pca_embeddings_test = np.concatenate((pca_embeddings_test, test_channel_pca), axis=1)


In [8]:
np.unique(train_genes)

array(['ARPC3', 'DGKE', 'FAN1', 'GAS7', 'HRAS', 'HSF1', 'HTT', 'IER5',
       'MAP4K4', 'MINK1', 'MLH1', 'MSH3', 'NCK1', 'NCKAP1', 'PGGT1B',
       'PPP2R1A', 'PPP2R2B', 'RAPGEF2', 'SLC25A11', 'SNX8', 'WASL',
       'non-target'], dtype='<U10')

In [9]:
# Binary classification: each gene vs non-target
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Make sure you have these variables defined from your previous code:
# train_labels, test_labels, X_train_scaled, X_test_scaled
# If not, you need to run the preprocessing steps first

# Prepare labels from gene names
train_labels = [gene[0] for gene in train_genes]
test_labels = [gene[0] for gene in test_genes]

# Train logistic regression model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(pca_embeddings_train)
X_test_scaled = scaler.transform(pca_embeddings_test)

# Get unique genes (excluding non-target)
unique_genes = list(set(train_labels + test_labels))
unique_genes = [gene for gene in unique_genes if gene != 'non-target']

print(f"Found {len(unique_genes)} unique genes for binary classification")
print("Genes:", unique_genes)

# Store results
binary_results = {}

for target_gene in unique_genes:
    
    # Create binary labels for training - only keep target gene and non-target samples
    train_binary_labels = []
    train_binary_indices = []
    for i, label in enumerate(train_labels):
        if label == target_gene:
            train_binary_labels.append('target')
            train_binary_indices.append(i)
        elif label == 'non-target':
            train_binary_labels.append('non-target')
            train_binary_indices.append(i)
    
    # Create binary labels for testing - only keep target gene and non-target samples
    test_binary_labels = []
    test_binary_indices = []
    for i, label in enumerate(test_labels):
        if label == target_gene:
            test_binary_labels.append('target')
            test_binary_indices.append(i)
        elif label == 'non-target':
            test_binary_labels.append('non-target')
            test_binary_indices.append(i)
    
    # Filter the feature matrices to only include relevant samples
    X_train_binary = X_train_scaled[train_binary_indices]
    X_test_binary = X_test_scaled[test_binary_indices]
    
    # Count samples
    train_target_count = sum(1 for label in train_binary_labels if label == 'target')
    train_nontarget_count = sum(1 for label in train_binary_labels if label == 'non-target')
    test_target_count = sum(1 for label in test_binary_labels if label == 'target')
    test_nontarget_count = sum(1 for label in test_binary_labels if label == 'non-target')

    
    # Skip if not enough samples in test set
    if test_target_count < 5:
        print(f"Skipping {target_gene} - too few test samples")
        continue
    
    # Train binary classifier
    binary_model = LogisticRegression(solver='saga', max_iter=1000, class_weight="balanced", random_state=42)
    binary_model.fit(X_train_binary, train_binary_labels)
    
    # Make predictions
    test_pred = binary_model.predict(X_test_binary)
    test_proba = binary_model.predict_proba(X_test_binary)
    
    # Calculate metrics
    precision, recall, f1, support = precision_recall_fscore_support(test_binary_labels, test_pred, average='binary', pos_label='target')
    
    # Calculate AUC
    try:
        # Get probability for the positive class (target)
        target_proba = test_proba[:, 1] if binary_model.classes_[1] == 'target' else test_proba[:, 0]
        auc = roc_auc_score([1 if label == 'target' else 0 for label in test_binary_labels], target_proba)
    except:
        auc = 0.0
    
    # Store results
    binary_results[target_gene] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'test_target_count': test_target_count,
        'test_nontarget_count': test_nontarget_count
    }


# Summary of results
print("\n" + "="*80)
print("BINARY CLASSIFICATION SUMMARY")
print("="*80)
print(f"{'Gene':<15} {'F1':<8} {'Precision':<10} {'Recall':<8} {'AUC':<8} {'Test Samples':<12}")
print("-"*80)

for gene, metrics in sorted(binary_results.items(), key=lambda x: x[1]['f1'], reverse=True):
    print(f"{gene:<15} {metrics['f1']:<8.4f} {metrics['precision']:<10.4f} {metrics['recall']:<8.4f} {metrics['auc']:<8.4f} {metrics['test_target_count']:<12}")

# Calculate average metrics
if binary_results:
    avg_f1 = np.mean([metrics['f1'] for metrics in binary_results.values()])
    avg_precision = np.mean([metrics['precision'] for metrics in binary_results.values()])
    avg_recall = np.mean([metrics['recall'] for metrics in binary_results.values()])
    avg_auc = np.mean([metrics['auc'] for metrics in binary_results.values()])
    
    print("-"*80)
    print(f"{'AVERAGE':<15} {avg_f1:<8.4f} {avg_precision:<10.4f} {avg_recall:<8.4f} {avg_auc:<8.4f}")

Found 19 unique genes for binary classification
Genes: ['MINK1', 'FAN1', 'WASL', 'NCKAP1', 'PPP2R2B', 'IER5', 'HSF1', 'NCK1', 'SNX8', 'HRAS', 'MAP4K4', 'MLH1', 'PPP2R1A', 'MSH3', 'SLC25A11', 'HTT', 'ARPC3', 'PGGT1B', 'RAPGEF2']
Skipping MSH3 - too few test samples

BINARY CLASSIFICATION SUMMARY
Gene            F1       Precision  Recall   AUC      Test Samples
--------------------------------------------------------------------------------
HRAS            0.9474   0.9000     1.0000   1.0000   9           
NCK1            0.8611   0.9394     0.7949   0.8237   39          
FAN1            0.8571   0.9231     0.8000   0.8375   30          
IER5            0.8462   0.9167     0.7857   0.9196   14          
RAPGEF2         0.8462   0.7857     0.9167   0.7396   12          
NCKAP1          0.8387   0.8125     0.8667   0.8083   15          
PPP2R2B         0.8000   0.9231     0.7059   0.7610   34          
SLC25A11        0.8000   0.8000     0.8000   0.8625   10          
SNX8            0.79

In [10]:
# Binary classification: each gene vs non-target
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Make sure you have these variables defined from your previous code:
# train_labels, test_labels, X_train_scaled, X_test_scaled
# If not, you need to run the preprocessing steps first

# Prepare labels from gene names
train_labels = [gene[0] for gene in train_genes]
test_labels = [gene[0] for gene in test_genes]

# Train random forest model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(pca_embeddings_train)
X_test_scaled = scaler.transform(pca_embeddings_test)

# Get unique genes (excluding non-target)
unique_genes = list(set(train_labels + test_labels))
unique_genes = [gene for gene in unique_genes if gene != 'non-target']

print(f"Found {len(unique_genes)} unique genes for binary classification")
print("Genes:", unique_genes)

# Store results
binary_results = {}

for target_gene in unique_genes:
    
    # Create binary labels for training - only keep target gene and non-target samples
    train_binary_labels = []
    train_binary_indices = []
    for i, label in enumerate(train_labels):
        if label == target_gene:
            train_binary_labels.append('target')
            train_binary_indices.append(i)
        elif label == 'non-target':
            train_binary_labels.append('non-target')
            train_binary_indices.append(i)
    
    # Create binary labels for testing - only keep target gene and non-target samples
    test_binary_labels = []
    test_binary_indices = []
    for i, label in enumerate(test_labels):
        if label == target_gene:
            test_binary_labels.append('target')
            test_binary_indices.append(i)
        elif label == 'non-target':
            test_binary_labels.append('non-target')
            test_binary_indices.append(i)
    
    # Filter the feature matrices to only include relevant samples
    X_train_binary = X_train_scaled[train_binary_indices]
    X_test_binary = X_test_scaled[test_binary_indices]
    
    # Count samples
    train_target_count = sum(1 for label in train_binary_labels if label == 'target')
    train_nontarget_count = sum(1 for label in train_binary_labels if label == 'non-target')
    test_target_count = sum(1 for label in test_binary_labels if label == 'target')
    test_nontarget_count = sum(1 for label in test_binary_labels if label == 'non-target')

    
    # Skip if not enough samples in test set
    if test_target_count < 5:
        print(f"Skipping {target_gene} - too few test samples")
        continue
    
    # Train random forest classifier
    binary_model = RandomForestClassifier(n_estimators=100, max_depth=25, class_weight="balanced", random_state=42, n_jobs=-1)
    binary_model.fit(X_train_binary, train_binary_labels)
    
    # Make predictions
    test_pred = binary_model.predict(X_test_binary)
    test_proba = binary_model.predict_proba(X_test_binary)
    
    # Calculate metrics
    precision, recall, f1, support = precision_recall_fscore_support(test_binary_labels, test_pred, average='binary', pos_label='target')
    
    # Calculate AUC
    try:
        # Get probability for the positive class (target)
        target_proba = test_proba[:, 1] if binary_model.classes_[1] == 'target' else test_proba[:, 0]
        auc = roc_auc_score([1 if label == 'target' else 0 for label in test_binary_labels], target_proba)
    except:
        auc = 0.0
    
    # Store results
    binary_results[target_gene] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'test_target_count': test_target_count,
        'test_nontarget_count': test_nontarget_count
    }


# Summary of results
print("\n" + "="*80)
print("BINARY CLASSIFICATION SUMMARY (Random Forest)")
print("="*80)
print(f"{'Gene':<15} {'F1':<8} {'Precision':<10} {'Recall':<8} {'AUC':<8} {'Test Samples':<12}")
print("-"*80)

for gene, metrics in sorted(binary_results.items(), key=lambda x: x[1]['f1'], reverse=True):
    print(f"{gene:<15} {metrics['f1']:<8.4f} {metrics['precision']:<10.4f} {metrics['recall']:<8.4f} {metrics['auc']:<8.4f} {metrics['test_target_count']:<12}")

# Calculate average metrics
if binary_results:
    avg_f1 = np.mean([metrics['f1'] for metrics in binary_results.values()])
    avg_precision = np.mean([metrics['precision'] for metrics in binary_results.values()])
    avg_recall = np.mean([metrics['recall'] for metrics in binary_results.values()])
    avg_auc = np.mean([metrics['auc'] for metrics in binary_results.values()])
    
    print("-"*80)
    print(f"{'AVERAGE':<15} {avg_f1:<8.4f} {avg_precision:<10.4f} {avg_recall:<8.4f} {avg_auc:<8.4f}")

Found 19 unique genes for binary classification
Genes: ['MINK1', 'FAN1', 'WASL', 'NCKAP1', 'PPP2R2B', 'IER5', 'HSF1', 'NCK1', 'SNX8', 'HRAS', 'MAP4K4', 'MLH1', 'PPP2R1A', 'MSH3', 'SLC25A11', 'HTT', 'ARPC3', 'PGGT1B', 'RAPGEF2']
Skipping MSH3 - too few test samples


/scr/vidit/conda_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



BINARY CLASSIFICATION SUMMARY (Random Forest)
Gene            F1       Precision  Recall   AUC      Test Samples
--------------------------------------------------------------------------------
NCK1            0.9070   0.8298     1.0000   0.8189   39          
MLH1            0.9032   1.0000     0.8235   0.9375   17          
SNX8            0.9024   0.8222     1.0000   0.6419   37          
MINK1           0.8966   0.8125     1.0000   0.8389   26          
PPP2R2B         0.8947   0.8095     1.0000   0.7316   34          
IER5            0.8889   0.9231     0.8571   0.9241   14          
FAN1            0.8824   0.7895     1.0000   0.8688   30          
MAP4K4          0.8621   0.7576     1.0000   0.5875   25          
HSF1            0.8571   0.7500     1.0000   0.4635   24          
RAPGEF2         0.8276   0.7059     1.0000   0.8594   12          
NCKAP1          0.7692   0.9091     0.6667   0.8000   15          
SLC25A11        0.6667   0.7500     0.6000   0.7312   10          
W